# Pipeline

A `Pipeline` lets you compose multiple `Step` objects into a reproducible workflow. Each step wraps a `Block`, `DataGenerator`, or plain callable. The pipeline can be executed locally or submitted to a SLURM cluster with a single method call.

## Directory structure

When a pipeline runs, it creates a directory hierarchy to organize artifacts:

```
rootdir/                     # Root directory (defaults to cwd)
└── <project_job>/           # job_dir = rootdir / project_job
    ├── .pipeline.pkl        # Serialized pipeline (SLURM mode)
    ├── logs/                # SLURM log files (SLURM mode)
    ├── slurm_scripts/       # Generated sbatch scripts (SLURM mode)
    └── <project_dir>/       # Step's project_dir (defaults to '.')
        ├── domain.json      # ExperimentData domain
        ├── input.csv        # ExperimentData input
        └── output.csv       # ExperimentData output
```

- **`rootdir`**: The parent directory where all job directories are created. Defaults to the current working directory.
- **`job_dir`** (`rootdir / project_job`): The directory for a specific pipeline run. The `project_job` is a unique identifier (defaults to a timestamp).
- **`project_dir`**: A relative sub-path within `job_dir` where `ExperimentData` is loaded and stored. Each `Step` has its own `project_dir` (defaults to `'.'`, i.e. the job directory itself). This is useful when different steps operate on different datasets.

## Creating Steps

A `Step` wraps a block (or callable) and gives it a name. The block can be:

- A `Block` subclass (operates on `ExperimentData`)
- A `DataGenerator` subclass (evaluates experiments, supports parallel execution)
- A plain callable with signature `(project_dir, **kwargs) -> None`

Plain callables are especially useful for the first step of a pipeline, where you need to create the `ExperimentData` from scratch.

Let's define a simple example. First, we create a callable that sets up an experiment, and a `Block` that doubles all output values.

In [ ]:
import numpy as np
from pathlib import Path

from f3dasm import (
    Block,
    ExperimentData,
    Pipeline,
    Step,
    Loop,
)
from f3dasm.design import Domain

In [ ]:
def create_experiment(project_dir: Path, n_samples: int = 10):
    """Create an ExperimentData with random samples and store it to disk."""
    domain = Domain()
    domain.add_float(name="x", low=0.0, high=1.0)
    domain.add_float(name="y", low=0.0, high=1.0)

    data = ExperimentData(domain=domain)
    data.sample(sampler="random", n_samples=n_samples, seed=42)

    # Compute a simple output: x + y
    x = data.to_numpy("input")
    data.add_output_numpy(x.sum(axis=1, keepdims=True), names=["sum"])

    data.store(project_dir=project_dir)
    print(f"Created experiment with {len(data)} samples at {project_dir}")

In [ ]:
class ScaleOutput(Block):
    """A Block that multiplies all output values by a given factor."""

    def __init__(self, factor: float = 2.0):
        self.factor = factor

    def call(self, data: ExperimentData, **kwargs) -> ExperimentData:
        output = data.to_numpy("output")
        scaled = output * self.factor
        data.add_output_numpy(scaled, names=["sum"])
        return data

## Building and running a Pipeline

Now we compose these into a `Pipeline` with two steps and run it locally:

In [ ]:
import tempfile

pipeline = Pipeline(
    name="example",
    steps=[
        Step(block=create_experiment, name="create", kwargs={"n_samples": 5}),
        Step(block=ScaleOutput(factor=3.0), name="scale"),
    ],
)

# Run locally in a temporary directory
with tempfile.TemporaryDirectory() as tmpdir:
    job_id = pipeline.run(mode="local", project_job="run_001", rootdir=tmpdir)
    print(f"Job ID: {job_id}")

    # Load and inspect the result
    result = ExperimentData.from_file(project_dir=Path(tmpdir) / job_id)
    print(result)

## Using Loops

A `Loop` repeats a group of steps for a given number of iterations. This is useful for iterative workflows like optimization loops, where you alternate between running experiments and updating a model.

Here we scale the output values three times in a row:

In [ ]:
pipeline_with_loop = Pipeline(
    name="loop_example",
    steps=[
        Step(block=create_experiment, name="create", kwargs={"n_samples": 5}),
        Loop(
            n_iterations=3,
            steps=[
                Step(block=ScaleOutput(factor=2.0), name="scale"),
            ],
        ),
    ],
)

with tempfile.TemporaryDirectory() as tmpdir:
    job_id = pipeline_with_loop.run(
        mode="local", project_job="run_002", rootdir=tmpdir
    )

    result = ExperimentData.from_file(project_dir=Path(tmpdir) / job_id)
    print("After 3x scaling by 2.0 (= 8x total):")
    print(result)

## Resuming from a step

If a pipeline run fails partway through, you can resume from a specific step using `from_step()`. Pass the same `project_job` to reuse the existing job directory:

In [ ]:
# Resume from the "scale" step, skipping "create"
resumed = pipeline.from_step("scale")
print(f"Resumed pipeline steps: {[s.name for s in resumed.steps]}")

## Running on SLURM

The same pipeline can be submitted to a SLURM cluster by switching the mode to `"slurm"` and providing a `SlurmCluster` configuration:

```python
from f3dasm import SlurmCluster, SlurmResources

cluster = SlurmCluster(
    partition="compute",
    account="my_account",
    env_setup=["module load python/3.11"],
)

pipeline.run(
    mode="slurm",
    cluster=cluster,
    rootdir="/scratch/user",
)
```

Each `Step` can specify its own `SlurmResources` (time, memory, CPUs). For pipelines with `Loop` elements, a self-resubmitting orchestrator script is generated to manage the iteration sequence.

You can preview the generated scripts without submitting using `generate_scripts()`:

```python
scripts = pipeline.generate_scripts(cluster=cluster)
for label, content in scripts.items():
    print(f"--- {label} ---")
    print(content)
```